# Fine-tune Stable Diffusion 1.5 (LoRA) trên ArtEmis + WikiArt

**Chuẩn bị trước trên Google Drive** (thư mục `MyDrive/artemis_sd/`):
- `Source/`: toàn bộ code của đồ án (copy cả thư mục Source lên Drive)
- `artemis_dataset_release_v0.csv`
- `kaggle.json` (API token Kaggle, để tải ảnh WikiArt) - hoặc tự đưa ảnh WikiArt lên Drive

Chọn **Runtime > Change runtime type > T4 GPU** trước khi chạy.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE   = '/content/drive/MyDrive/artemis_sd'
SRC    = f'{BASE}/Source'
CSV    = f'{BASE}/artemis_dataset_release_v0.csv'
OUT    = f'{BASE}/outputs'            # checkpoint + kết quả lưu trên Drive để không mất khi ngắt phiên
DATA   = '/content/data'              # dữ liệu đã xử lý, để trên ổ local cho nhanh
WIKI   = '/content/wikiart'

%cd {SRC}

In [ ]:
# gỡ torchao: bản cài sẵn trên Colab quá cũ khiến peft báo "incompatible version of torchao" (dự án không dùng)
!pip uninstall -y -q torchao
!pip install -q diffusers transformers peft accelerate bitsandbytes tqdm pandas gradio
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU')

## 1. Tải ảnh WikiArt (bỏ qua nếu đã có ảnh)

In [ ]:
import os
if not os.path.isdir(WIKI):
    os.makedirs('/root/.kaggle', exist_ok=True)
    !cp {BASE}/kaggle.json /root/.kaggle/kaggle.json && chmod 600 /root/.kaggle/kaggle.json
    !pip install -q kaggle
    !kaggle datasets download -d steubk/wikiart -p {WIKI} --unzip
!ls {WIKI} | head

## 2. Chuẩn bị dữ liệu (xử lý 1 lần, nén lại lên Drive để lần sau dùng lại)

In [ ]:
import os
ZIP = f'{BASE}/prepared_data.zip'
if os.path.exists(ZIP):
    !unzip -q -o {ZIP} -d {DATA}
else:
    !python prepare_data.py --csv {CSV} --images-root {WIKI} --out {DATA} --num-paintings 8000
    !cd {DATA} && zip -q -r {ZIP} .

## 3. Huấn luyện (có thể chạy lại cell này sau khi bị ngắt phiên, `--resume` sẽ tiếp tục)

In [ ]:
!python train_lora.py --data_dir {DATA} --output_dir {OUT}/lora --max_steps 6000 \
    --batch_size 1 --grad_accum 4 --lr 1e-4 --lora_rank 16 --mixed_precision fp16 \
    --use_8bit_adam --save_every 500 --sample_every 500 --resume

## 4. Xem ảnh mẫu và đường cong loss

In [ ]:
import glob, pandas as pd
from PIL import Image
from IPython.display import display
import matplotlib.pyplot as plt

df = pd.read_csv(f'{OUT}/lora/loss_log.csv')
plt.plot(df.step, df.loss); plt.xlabel('step'); plt.ylabel('loss'); plt.show()

for p in sorted(glob.glob(f'{OUT}/lora/samples/*.png'))[-4:]:
    display(Image.open(p))

## 5. Đánh giá (CLIP score: SD gốc vs SD + LoRA)

In [ ]:
!python evaluate.py --data_dir {DATA} --lora_dir {OUT}/lora/final --num_prompts 100 --out {OUT}/eval
import json; print(json.dumps(json.load(open(f'{OUT}/eval/summary.json')), indent=2))
display(Image.open(f'{OUT}/eval/compare_grid.png'))
display(Image.open(f'{OUT}/eval/emotion_grid.png'))

## 6. (Tùy chọn) Mở Web UI demo

In [ ]:
!python app.py --lora_dir {OUT}/lora/final --share